In [19]:
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install faiss-cpu


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import os
import pickle
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

## Load Chunked Documents

In [21]:
CHUNK_PATH = "../data/document_chunks.pkl"

with open(CHUNK_PATH, "rb") as file:
    chunks = pickle.load(file)

print(f"Loaded {len(chunks)} chunks.")

Loaded 494 chunks.


In [22]:
chunks[0]

Document(metadata={'source': 'FAQ', 'category': 'Order Tracking'}, page_content='Question:\nCan I track multiple orders?\n\nAnswer:\nYes, each order has its own tracking information in your account.')

In [23]:
print(chunks[0].page_content)

Question:
Can I track multiple orders?

Answer:
Yes, each order has its own tracking information in your account.


## Embedding Model

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6209.76it/s]


## Embeddings and Build FAISS

In [25]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
print("FAISS Vector Database Created Successfully")

FAISS Vector Database Created Successfully


In [26]:
print("Number of Indexed Chunks:", vectorstore.index.ntotal)

Number of Indexed Chunks: 494


## Save Vector Database

In [27]:
VECTOR_PATH = "../vectorstore/faiss_index"
vectorstore.save_local(VECTOR_PATH)
print("FAISS Database Saved Successfully")

FAISS Database Saved Successfully


## Reload the Database

In [28]:
db = FAISS.load_local(
    VECTOR_PATH,
    embedding_model,
    allow_dangerous_deserialization=True
)
print("FAISS Database Loaded Successfully")

FAISS Database Loaded Successfully


## Test Semantic Retrieval

In [29]:
query = "How do I return a product?"
results = db.similarity_search(query, k=3)
print("Retrieved Documents:", len(results))

Retrieved Documents: 3


In [30]:
for i, doc in enumerate(results):
    print("="*80)
    print(f"Result {i+1}")
    print()
    print(doc.page_content)

Result 1

Question:
How do I return a product? (20)

Answer:
Initiate a return from your order history within the eligible return window.
Result 2

Question:
How do I return a product? (6)

Answer:
Initiate a return from your order history within the eligible return window.
Result 3

Question:
How do I return a product? (12)

Answer:
Initiate a return from your order history within the eligible return window.


## Test Multiple Queries

In [31]:
queries = [
    "How can I reset my password?",
    "When will I receive my refund?",
    "How do I claim warranty?",
    "Can I pay using UPI?",
    "How do I contact technical support?"
]
for q in queries:
    print("="*80)
    print("Query:", q)
    docs = db.similarity_search(q, k=1)
    print(docs[0].page_content)
    print()

Query: How can I reset my password?


Question:
How do I reset my password? (10)

Answer:
Use the 'Forgot Password' option on the login page.

Query: When will I receive my refund?
Question:
When will I receive my refund? (18)

Answer:
Refunds are typically processed within 5 to 7 business days after approval.

Query: How do I claim warranty?
Question:
How do I claim warranty? (10)

Answer:
Provide proof of purchase and submit a warranty request.

Query: Can I pay using UPI?
Question:
Can I pay with UPI? (27)

Answer:
Yes, UPI payments are supported.

Query: How do I contact technical support?
Question:
How do I contact technical support?

Answer:
Technical support is available through chat, email, and phone.



## Using a Retriever

In [32]:
retriever = db.as_retriever(
    search_kwargs={"k":3}
)
print("Retriever Created Successfully")

Retriever Created Successfully


In [33]:
query = "What payment methods are accepted?"
documents = retriever.invoke(query)
print("Retrieved Documents:", len(documents))

Retrieved Documents: 3


In [34]:
for doc in documents:
    print("="*80)
    print(doc.page_content)

Question:
What payment methods are accepted? (47)

Answer:
We accept UPI, debit cards, credit cards, net banking, and supported wallets.
Question:
What payment methods are accepted? (6)

Answer:
We accept UPI, debit cards, credit cards, net banking, and supported wallets.
Question:
What payment methods are accepted? (36)

Answer:
We accept UPI, debit cards, credit cards, net banking, and supported wallets.


## Verify Metadata

In [35]:
documents[0].metadata

{'source': 'FAQ', 'category': 'Payments'}